# La cara como llave de la identidad digital europea: la medición

Cuaderno de lectura de la medición `biometria-hf/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Lee la instantánea del 26 de noviembre de 2026 que escribe `generar.py`: recuentos de las aplicaciones (Spaces) y los modelos de Hugging Face que sirven para intercambiar caras frente a los que detectan la falsificación, y lo que mencionan las fichas de los detectores. Solo metadatos públicos: ningún modelo se descarga y ninguna imagen de ninguna persona interviene. Entrada: [manpla.net](https://manpla.net/posts/cara-identidad-digital/).

In [ ]:
import gzip, io, json, urllib.request
from pathlib import Path
import pandas as pd

TOMA = "instantaneas/2026-11-26/"
RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/biometria-hf/" + TOMA

def abrir(nombre):
    """el fichero de la instantánea local si existe; si no, el del repositorio."""
    p = Path(TOMA) / nombre
    return p.read_bytes() if p.exists() else urllib.request.urlopen(RAW + nombre).read()

def leer(nombre, **kw):
    return pd.read_csv(io.BytesIO(abrir(nombre)), **kw)

resumen = json.loads(abrir("resumen.json"))
print("consulta:", resumen["fecha_consulta"])
print("repositorios por tipo y familia:")
for k, v in sorted(resumen["apartados"].items()):
    print(f"  {k:22s} {v:5d}")

## Recuentos por familia, subtipo y modalidad

In [ ]:
r = leer("recuentos.csv")
print(r.sort_values(["tipo", "familia", "subtipo", "modalidad"]).to_string(index=False))

## Las tres poblaciones de la entrada: intercambio, detección facial (manipulación y vida) y voz

In [ ]:
def pob(t):
    g = r[r.tipo == t]
    filas = {
        "intercambio y morphing": g[g.familia == "generación"],
        "manipulación facial": g[(g.familia == "detección") & (g.modalidad == "imagen y vídeo") & g.subtipo.isin(["deepfake", "morphing"])],
        "presentación y vida": g[(g.familia == "detección") & (g.subtipo == "presentación y vida")],
        "voz": g[(g.familia == "detección") & (g.modalidad == "voz")]}
    return pd.DataFrame({k: v[["repos", "descargas_total", "spaces_en_marcha"]].sum() for k, v in filas.items()}).T

print("aplicaciones (Spaces)")
print(pob("space")[["repos", "spaces_en_marcha"]].to_string())
print("\nmodelos")
print(pob("modelo")[["repos", "descargas_total"]].to_string())

## Qué mencionan las fichas de los modelos detectores

In [ ]:
for k, v in resumen["fichas"].items():
    print(f"  {k:18s} {v['si']:4d} de {v['n']}")
print("detectores de vida:")
for k, v in resumen["fichas_vida"].items():
    print(f"  {k:18s} {v['si']:4d} de {v['n']}")

## Aplicaciones creadas por trimestre (repositorios vigentes el día de la consulta)

In [ ]:
s = leer("serie_trimestral.csv")
s = s[(s.tipo == "space") & (s.trimestre >= "2025")]
print(s.pivot_table(index="trimestre", columns="familia", values="repos", fill_value=0).to_string())

## Los metadatos completos de la toma

In [ ]:
meta = json.loads(gzip.decompress(abrir("metadatos.json.gz")))
print(len(meta), "repositorios con sus metadatos de la API")
tipos = pd.Series([m["tipo_api"] for m in meta]).value_counts().sort_index().sort_values(ascending=False, kind="stable")
print(tipos.to_string())